# **Raw Data Preparation for Pre-train the BERT Model**

#### **Import Libraries**

In [63]:
import torch.nn as nn
from torchtext.vocab import build_vocab_from_iterator
from torchtext.data.utils import get_tokenizer
from transformers import BertTokenizer
from torchtext.datasets import IMDB
import random
import torch

In [2]:
tokenizor = BertTokenizer.from_pretrained("bert-base-uncased")

In [11]:
text = "transformers are powerful models. I am playing cricket"
tokens = tokenizor.tokenize(text)
tokens

['transformers',
 'are',
 'powerful',
 'models',
 '.',
 'i',
 'am',
 'playing',
 'cricket']

In [16]:
#Dataset unpack
train_iter, val_iter = IMDB()
train_iter

ShardingFilterIterDataPipe

In [17]:
def yield_tokens(data_iter):
    for label, text in data_iter:
        yield tokenizor.tokenize(text)

In [62]:
PAD_IDX, CLS_IDX, SEP_IDX, MASK_IDX, UNK_IDX = 0,1,2,3,4
special_tokens = ['PAD_IDX','CLS_IDX', 'SEP_IDX', 'MASK_IDX', 'UNK_IDX']


vocab = build_vocab_from_iterator(yield_tokens(train_iter),
                                  specials= special_tokens,
                                  special_first=True, max_tokens=1000)
vocab.set_default_index(UNK_IDX)

vocab_size = len(vocab)

In [60]:
vocab(tokens)
vocab.get_itos()[53]


'there'

### **Text Masking** 

In [22]:
def bernoulli_true_false (percentage):
    number:bool = random.random() < percentage
    return number
    

In [61]:
def index_to_text(index):
    return vocab.get_itos()[index]

In [64]:
def text_masking(token):
    mask = bernoulli_true_false(0.2)
    
    token_ = ""
    mask_label = ""
    
    if not mask:
        token_ = token
        mask_label = '[PAD]'# if mask false then return as it is and the label also
    
    random_opp = bernoulli_true_false(0.5)
    random_switch = bernoulli_true_false(0.5)
    
    # case 1: if mask is true, random_opp is true and random_sqitch is true
    if mask and random_opp and random_switch:
        token_ = '[MASK]'
        mask_label =  index_to_text(torch.randint(0, vocab_size,(1,)))
        
    #case 2: if mask is true, random_opp is true and random_switch is false
    elif mask and random_opp and not random_switch:
        token_ = token
        mask_label = token
        
    #case 3: if mask is true and random_opp is false
    elif mask and not random_opp:
        token_ = '[MASK]'
        mask_label = token
    
    return token_, mask_label
    
    

In [151]:
text_masking('playing')

('[MASK]', '##io')

### **MLM Preparation**

In [210]:
def mlm_preparation (tokens, include_raw_tokens = False):
    
    bert_input = []
    bert_label = []
    raw_tokens = []
    bert_current_input = []
    bert_current_label = []
    current_raw_tokens = []
    
    for token in tokens:
        
        mask_token, mask_label = text_masking(token)
        
        bert_current_input.append(mask_token)
        bert_current_label.append(mask_label)
        
        if include_raw_tokens:
            current_raw_tokens.append(token)
            
        # check whether the tokens is sentence delimeter
        if token in ['.','?','!']:
            
            if len(bert_current_input) >  2:
                bert_input.append(bert_current_input)
                bert_label.append(bert_current_label)
                
                if include_raw_tokens:
                    raw_tokens.append(current_raw_tokens)
                
                # reset the list for next sentence
                bert_current_input = []
                bert_current_label = []
                current_raw_tokens = []
            
            # length is not enough to create a sentence
            else:
                bert_current_input = []
                bert_current_label = []
                current_raw_tokens = []               
            
    if bert_current_input:
        bert_input.append(bert_current_input)
        bert_label.append(bert_current_label)
            
        if include_raw_tokens:
                raw_tokens.append(current_raw_tokens)       
        
    return (bert_input, bert_label, raw_tokens) if include_raw_tokens else (bert_input,bert_label)
    
    

In [211]:
bert_input, bert_label = mlm_preparation(tokens) 
bert_input

[['transformers', 'are', '[MASK]', 'models', '.'],
 ['i', 'am', '[MASK]', 'cricket']]

In [212]:
bert_label

[['[PAD]', '[PAD]', 'career', '[PAD]', '[PAD]'],
 ['[PAD]', '[PAD]', 'new', '[PAD]']]

### **Function for Next Sentence Prediction**

In [213]:
# in here we are looking to evaluate and comparison between bert_input and bert_label 
def nsp_preparation (input_sentence, input_mask_label):
    
    if len(input_sentence) < 2:
        raise ValueError("input sentence size must be larger than 2")
    
    if (len(input_sentence) == len(input_mask_label)):
        raise ValueError ("input sentence and input mask label must have the same length")

    bert_input, bert_label, is_next = [], [], []
    
    available_indices = list(range(len(input_sentence)))
    
    while len(available_indices) >= 2:
        
        if random.random() < 0.5:
            index = random.choice(available_indices[:-1])
           
            bert_input.append([['[CLS]'] + input_sentence[index]+['[SEP]'],
                            input_sentence[index+1]+['[SEP]']])
            
            bert_label.append([['[PAD]'] + input_mask_label[index] +['[PAD]'],
                               input_mask_label[index +1]+ ['[PAD]']])
            
            is_next.append(1) # 1 label indicate two sentences are consecutive
            
            available_indices.remove(index)
            if (index+1) in available_indices:
                available_indices.remove(index+1)
        
        else:
            
            indices = random.sample(available_indices, 2)
            
            bert_input.append([['[CLS]'] + input_sentence[indices[0]] + ['[SEP]'],
                               input_sentence[indices[1]]+ ['[SEP]']])
            
            bert_label.append([['[PAD]'] + input_mask_label[indices[0]]+ ['[PAD]'],
                               input_mask_label[indices[1]]+['[PAD]']])
            
            is_next.append(0)
            
            available_indices.remove(indices[0])
            available_indices.remove(indices[1])
    
    return bert_input, bert_label, is_next 